In [ ]:
# cub-200 데이터셋을 위한 분류 과제
# 1. 미세분류 -> 전이학습 모델은 관여 안함(모델은 자유롭게)
# 2. 성능확인, 모델
# 3. 성능평가지표, top1, top5, top10

In [ ]:
import torch
import os
import shutil
import glob
from sklearn.model_selection import train_test_split
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim

USE_MPS = torch.backends.mps.is_available()
DEVICE = torch.device('mps' if USE_MPS else 'cpu')

In [ ]:
original_dataset_dir = '../../data/archive/CUB_200_2011/images'
base_dir = Path('../../data/CUB_200_2011_split')
if not os.path.exists(base_dir):
    os.mkdir(base_dir)

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

if not os.path.exists(train_dir):
    os.mkdir(train_dir)
if not os.path.exists(validation_dir):
    os.mkdir(validation_dir)
if not os.path.exists(test_dir):
    os.mkdir(test_dir)

breeds = os.listdir(original_dataset_dir)

for breed_folder in breeds:
    train_breed_dir = os.path.join(train_dir, breed_folder)
    validation_breed_dir = os.path.join(validation_dir, breed_folder)
    test_breed_dir = os.path.join(test_dir, breed_folder)

    if not os.path.exists(train_breed_dir):
        os.mkdir(train_breed_dir)
    if not os.path.exists(validation_breed_dir):
        os.mkdir(validation_breed_dir)
    if not os.path.exists(test_breed_dir):
        os.mkdir(test_breed_dir)

    src_dir = os.path.join(original_dataset_dir, breed_folder)
    image_files = glob.glob(os.path.join(src_dir, '*.jpg'))

    train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42, shuffle=True)
    val_files, test_files = train_test_split(val_files, test_size=0.5, random_state=42, shuffle=False)

    for f in train_files:
        shutil.copy(f, os.path.join(train_breed_dir, os.path.basename(f)))

    for f in val_files:
        shutil.copy(f, os.path.join(validation_breed_dir, os.path.basename(f)))

    for f in test_files:
        shutil.copy(f, os.path.join(test_breed_dir, os.path.basename(f)))

print(f"데이터 분할 완료! '{base_dir}' 폴더를 확인해주세요.")

In [ ]:
#데이터 로더 구조 정의
class Make_dataset_Transform:
    def __init__(self, resize, mean, std):
        self.base_transform = {
            'train': transforms.Compose([
                #데이터 증강
                transforms.RandomResizedCrop(resize, scale=(0.5, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                transforms.RandomRotation(15),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ]),
            'val': transforms.Compose([
                transforms.Resize(resize),
                transforms.CenterCrop(resize),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ])
        }

    def __call__(self, phase='train'):
        return self.base_transform[phase]

In [ ]:
#상수 정의
resize = 380
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
BATCH_SIZE = 64

#데이터 로더 정의
transform = Make_dataset_Transform(resize, mean, std)
base_dir = Path('../../data/CUB_200_2011_split')

tr_ds = datasets.ImageFolder(base_dir / 'train', transform=transform('train'))
val_ds = datasets.ImageFolder(base_dir / 'validation', transform=transform('val'))
test_ds = datasets.ImageFolder(base_dir / 'test', transform=transform('val'))

tr_ds_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
val_ds_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_ds_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
tr_x, tr_y = next(iter(tr_ds_loader))
val_x, val_y = next(iter(val_ds_loader))

In [ ]:
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
efficientnet_v2_s = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)

In [ ]:
for p in efficientnet_v2_s.features.parameters():
    p.requires_grad = False

In [ ]:
class_n = len(tr_ds.classes)
classifier_in_n = efficientnet_v2_s.classifier[-1].in_features
efficientnet_v2_s.classifier[-1] = nn.Linear(classifier_in_n, class_n)

In [ ]:
def run_epoch(m, loder, optimizer, train=True):
    m.train(train)
    total_loss, total_correct, total = 0.0, 0, 0
    for x, y in loder:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            output = m(x)
            loss = criterion(output, y)
            if train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (output.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, total_correct / total

def evaluate(m, loader):
    m.eval()
    total_correct_1, total_correct_5, total_correct_10 = 0, 0, 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            output = m(x)
            _, preds = torch.topk(output, 10, dim=1)
            y_resized = y.view(-1, 1)

            total_correct_1 += (preds[:, :1] == y_resized).sum().item()
            total_correct_5 += (preds[:, :5] == y_resized).sum().item()
            total_correct_10 += (preds[:, :10] == y_resized).sum().item()
            total += x.size(0)

    # 정확도 계산
    acc1 = total_correct_1 / total
    acc5 = total_correct_5 / total
    acc10 = total_correct_10 / total

    return acc1, acc5, acc10

In [ ]:
efficientnet_v2_s = efficientnet_v2_s.to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
num_epochs = 100
best_val_acc = 0.0
model_save_path = './best_efficient-b4.pth'

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# 1단계: 분류기만 학습
print("--- 1단계: 분류기 학습 시작 ---")
for p in efficientnet_v2_s.features.parameters():
    p.requires_grad = False

opt_classifier = optim.Adam(efficientnet_v2_s.classifier.parameters(), lr=0.001)
scheduler_classifier = ReduceLROnPlateau(opt_classifier, mode='min', factor=0.1, patience=3)

# 1단계 학습 루프
for epoch in range(50):
    tr_loss, tr_acc = run_epoch(efficientnet_v2_s, tr_ds_loader, opt_classifier, train=True)
    val_loss, val_acc = run_epoch(efficientnet_v2_s, val_ds_loader, opt_classifier, train=False)
    scheduler_classifier.step(val_loss)
    print(f'Epoch {epoch + 1}/{num_epochs} | tr_loss: {tr_loss:.4f}, tr_acc: {tr_acc:.4f} | val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(efficientnet_v2_s.state_dict(), model_save_path)
        print(f"최고의 모델이 저장되었습니다! '{model_save_path}' (Val Acc: {best_val_acc:.4f})")

# 2단계: 전체 미세 조정
print("\\n--- 2단계: 전체 미세 조정 시작 ---")
for p in efficientnet_v2_s.parameters():
    p.requires_grad = True

opt_finetune = optim.Adam([
    {'params': efficientnet_v2_s.features.parameters(), 'lr': 1e-5},
    {'params': efficientnet_v2_s.classifier.parameters(), 'lr': 1e-4}
])
scheduler_finetune = ReduceLROnPlateau(opt_finetune, mode='min', factor=0.1, patience=5)

# 2단계 학습 루프
for epoch in range(50, num_epochs):
    tr_loss, tr_acc = run_epoch(efficientnet_v2_s, tr_ds_loader, opt_finetune, train=True)
    val_loss, val_acc = run_epoch(efficientnet_v2_s, val_ds_loader, opt_finetune, train=False)
    scheduler_finetune.step(val_loss)
    print(f'Epoch {epoch + 1}/{num_epochs} | tr_loss: {tr_loss:.4f}, tr_acc: {tr_acc:.4f} | val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(efficientnet_v2_s.state_dict(), model_save_path)
        print(f"최고의 모델이 저장되었습니다! '{model_save_path}' (Val Acc: {best_val_acc:.4f})")

In [ ]:
print("\n--- 최종 테스트 평가 ---")
best_model = efficientnet_b4(weights=None)
best_model.classifier[-1] = nn.Linear(best_model.classifier[-1].in_features, len(tr_ds.classes))
best_model.load_state_dict(torch.load(model_save_path))
best_model.to(DEVICE)

test_acc1, test_acc5, test_acc10 = evaluate(best_model, test_ds_loader)
print(f"  Top-1 Accuracy: {test_acc1:.4f}")
print(f"  Top-5 Accuracy: {test_acc5:.4f}")
print(f"  Top-10 Accuracy: {test_acc10:.4f}")